# Avaliação clínica: `detalhe` vs `detalhe_{model}`

Este notebook reproduz o job batch [`pipelines/clinical/events_detalhe_eval.py`](../../pipelines/clinical/events_detalhe_eval.py) usando a API importável `wokibi_eval.clinical`.

## Objetivo

Comparar o texto original (`detalhe`) com a versão gerada por um modelo (`detalhe_gpt_oss_120b`, etc.) via `ClinicalEvaluator` e gravar métricas em `results/clinical/<dataset>/<data_version>/`.

## Pré-requisitos

1. Na raiz do repo: `poetry install --with clinical --with dev`
2. Modelo spaCy: `poetry run python -m spacy download pt_core_news_lg`
3. `.env` com `MONGODB_*` (Mongo) e `GOOGLE_API_KEY` se `ENABLE_JUDGE=True`
4. Kernel Jupyter = Python do `.venv` deste projeto

**Nota:** `--model_name` avalia o campo processado `detalhe_{model}`. O juiz LLM usa `LLM_MODEL` do wokibi-data ou `JUDGE_MODEL` abaixo. Com `cliid` publicado no venv, subir o juiz Gemini pode exigir variáveis Infobip; use `ENABLE_JUDGE=False` para só métricas locais ou `./scripts/use-local-wokibi.sh link all`.

## Configuração da run

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from wokibi_ai.llm import LLM
from wokibi_ai.llm_fields import detalhe_field_name, sanitize_model_name
from wokibi_data.config import LLM_MODEL, partner_parameters
from wokibi_eval.clinical import EventsDetalheEvaluator
from wokibi_eval.evaluation.clinical_evaluator import ClinicalEvaluator
from wokibi_eval.paths import clinical_detalhe_eval_dir

In [ ]:
DATASET = "uti"
MODEL_NAME = "gpt-oss-120b"
TIPO_EVENTO = "evolução clínica (internação)"  # None = todos os tipos elegíveis
ENABLE_JUDGE = True
JUDGE_MODEL = None  # None → LLM_MODEL do wokibi-data

data_version = partner_parameters[DATASET]["data_version"]
out_dir = clinical_detalhe_eval_dir(DATASET, data_version)
csv_path = out_dir / f"detalhe_{sanitize_model_name(MODEL_NAME)}_eval.csv"

print("data_version:", data_version)
print("campo processado:", detalhe_field_name(MODEL_NAME))
print("CSV:", csv_path)
print("juiz default (wokibi-data):", LLM_MODEL)

## Passo 1 — Smoke test (par sintético, sem Mongo)

Pilares do `ClinicalEvaluator`:

| Grupo | Colunas principais |
|-------|-------------------|
| Compressão | `compression_rate`, `word_compression_rate`, `is_empty_processed`, `is_identical` |
| Números/doses | `numeric_preservation_rate` |
| NER | `ner_recall`, `ner_precision`, `ner_f1_score` |
| Semântica | `biobert_similarity` |
| Juiz (opcional) | `score_faithfulness`, `score_completeness`, `score_readability` (1–5) |

In [ ]:
judge_llm = None
if ENABLE_JUDGE:
    judge_llm = LLM(JUDGE_MODEL or LLM_MODEL)
    judge_llm.start_llm()

evaluator = ClinicalEvaluator(judge_llm=judge_llm, enable_judge=ENABLE_JUDGE)

original = "Paciente com PA 120x80, FC 88, em uso de losartana 50 mg VO."
processed = "PA 120/80, FC 88. Losartana 50 mg VO."

pd.Series(evaluator.evaluate(original, processed))

## Passo 2 — Um evento do Mongo (smoke batch)

Eventos já presentes no CSV (mesmo `event_id`) são ignorados (idempotência).

In [ ]:
EventsDetalheEvaluator(
    DATASET,
    [MODEL_NAME],
    enable_judge=ENABLE_JUDGE,
    judge_model=JUDGE_MODEL,
).run(
    verbose=True,
    break_on=True,
    tipo_evento=TIPO_EVENTO,
    max_events=1,
)

## Passo 3 — Amostra maior

Equivalente a `break_on=False` e `max_events` na CLI.

In [ ]:
MAX_EVENTS = 33  # ajuste ou comente a célula se não quiser rodar

EventsDetalheEvaluator(
    DATASET,
    [MODEL_NAME],
    enable_judge=ENABLE_JUDGE,
    judge_model=JUDGE_MODEL,
).run(
    verbose=True,
    break_on=False,
    tipo_evento=TIPO_EVENTO,
    max_events=MAX_EVENTS,
)

## Passo 4 — Analisar o CSV

In [ ]:
if not csv_path.exists():
    raise FileNotFoundError(f"Rode os passos anteriores ou o CLI. Esperado: {csv_path}")

df = pd.read_csv(csv_path)
df.head()

In [ ]:
metric_cols = [
    "compression_rate",
    "numeric_preservation_rate",
    "ner_f1_score",
    "biobert_similarity",
    "score_faithfulness",
    "score_completeness",
    "score_readability",
]
present = [c for c in metric_cols if c in df.columns]
df[present].agg(["mean", "median", "std"]).T

## Passo 5 — Outliers (ex.: juiz baixo + BioBERT alto)

In [ ]:
if "score_faithfulness" in df.columns and "biobert_similarity" in df.columns:
    df.query("score_faithfulness < 3 and biobert_similarity > 0.9")[
        ["event_id", "tipo_evento"] + present
    ]

## Anexo — CLI equivalente

```bash
cd wokibi-eval
poetry run python pipelines/clinical/events_detalhe_eval.py \
  --dataset=uti \
  --tipo_evento='evolução clínica (internação)' \
  --model_name=gpt-oss-120b \
  --judge_model=gemini-2.5-flash \
  --verbose=True \
  --break_on=False \
  --max_events=33
```